In [52]:
!pip install -q -U google-generativeai

In [53]:
import time
import json
import warnings
import pandas as pd

import google.generativeai as genai
from google.colab import userdata
from google.api_core import exceptions

In [54]:
import logging
logging.getLogger("tornado.access").setLevel(logging.CRITICAL)

In [55]:
warnings.filterwarnings('ignore')

In [56]:
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

In [57]:
genai_model = genai.GenerativeModel('gemini-2.5-flash-lite')

#### Load the data from a json file.

In [58]:
with open('Data/data.json', 'rt') as f_in:
    documents = json.load(f_in)

#### Template for all the prompts to generate questions for data entries.

In [59]:
prompt_templet = """

    You're a friendly and knowledgeable filmmaking tutor! Your task is to transform a single filmmaking concept into 5 engaging, beginner-friendly questions. These questions should help a student truly understand and apply the concept.

    Here is the concept you'll be working with, pulled from our filmmaking knowledge base:

    Type: {type}
    Term: {term}
    Definition: {definition}
    Extra: {extra}

    Please follow these guidelines when generating your questions:

    - Include a variety of question types:
    - At least one recall question (e.g., "What is...?")
    - At least one application question (e.g., "When would you use...?")
    - Keep the questions short, clear, and easy to understand.
    - Only provide the questions—do not include answers.
    - Format the output as a JSON array of strings.
    - Do not use any text formatting, such as bold or italics.
    - Do not use code blocks in your output.
    - Output should be json parsable.


""".strip()

#### Check if google api working or not.

In [9]:
temp_question = "Hello!"
temp_response = genai_model.generate_content(temp_question)
print(temp_response.text)

Hello! How can I help you today?


#### Generating Questions.

In [60]:
done = 0
MAX_RETRIES = 5
DELAY_SECONDS = 5
response = {}

In [61]:
for record in documents:

    if record['id'] in response:
        continue

    for attempt in range(MAX_RETRIES):

        try:
            prompt = prompt_templet.format(**record)
            questions = genai_model.generate_content(prompt)
            response[record['id']] = questions.text
            break

        except exceptions.TooManyRequests as e:
            print(f"Too many requests made. Trying again after 15 seconds...")
            time.sleep(15)

        except exceptions.InternalServerError as e:
            print(f'Internal server error: Trying again..... {attempt + 1}/{MAX_RETRIES}')
            time.sleep(DELAY_SECONDS * (2 ** attempt))

        except Exception as e:
            print(f"An unexpected error occurred. Skipping... Error: {e}")
            break

    done += 1
    if done % 15 == 0:
        print(f'Records Completed: {done}/{len(documents)}')
        time.sleep(30)

print('Process Completed!')

Records Completed: 15/829
Records Completed: 30/829
Internal server error: Trying again..... 1/5
Internal server error: Trying again..... 1/5
Records Completed: 45/829
Records Completed: 60/829
Records Completed: 75/829
Internal server error: Trying again..... 1/5
Records Completed: 90/829
Records Completed: 105/829
Records Completed: 120/829
Records Completed: 135/829
Records Completed: 150/829
Records Completed: 165/829
Records Completed: 180/829
Records Completed: 195/829
Records Completed: 210/829
Internal server error: Trying again..... 1/5
Records Completed: 225/829
Internal server error: Trying again..... 1/5
Records Completed: 240/829
Records Completed: 255/829
Records Completed: 270/829
Too many requests made. Trying again after 15 seconds...
Records Completed: 285/829
Records Completed: 300/829
Too many requests made. Trying again after 15 seconds...
Records Completed: 315/829
Internal server error: Trying again..... 1/5
Records Completed: 330/829
Internal server error: Tryin

In [62]:
len(response)

829

In [63]:
if len(response) != len(documents):

    for record in documents:

        if record['id'] not in response:
            prompt = prompt_templet.format(**record)
            questions = genai_model.generate_content(prompt)
            response[record['id']] = questions.text

print(f'response entries: {len(response)}')
print(f'   total records: {len(documents)}')

response entries: 829
   total records: 829


In [115]:
def fix_json_output(text):

    questions = []

    for que in text.split('\n')[1:6]:
        cleaned = que[:-1].strip().strip('"')
        questions.append(cleaned)

    return questions


In [117]:
data = []

for id, questions in response.items():

    for question in fix_json_output(questions):

        row = {'question': question, 'id': id}
        data.append(row)

In [122]:
ground_truth = pd.DataFrame(data)

In [123]:
ground_truth.to_csv(
    'ground_truth.csv',
    index = False
)